# Chapter 7: Search In-Depth
## Topic 7: Reranking — Cross-Encoders

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every retrieval method covered so far — BM25, dense retrieval, hybrid RRF — is what's called a bi-encoder approach: the query and each document are encoded completely independently into their own representations, and similarity is computed afterward with a simple, cheap operation (a dot product, a sum of term scores).
- This independence is exactly what makes these methods fast enough to search an entire corpus: document representations are computed once, ahead of time, at ingest time, and only the incoming query needs to be processed at search time.
- The cost of this independence: the model never actually sees the query and a specific document together at the same time. It can only compare two representations that were each computed without any knowledge of the other — it cannot model fine, specific interactions between particular words in the query and particular words in a specific document.
- A cross-encoder removes this limitation entirely by taking the query and one candidate document together, as a single combined input, letting the model directly compare every part of the query against every part of that specific document — producing a single, much more accurate relevance score for that pair.
- The cost: a cross-encoder can't precompute anything ahead of time, since it needs both the query and the document together as input. Every single (query, document) pair requires a full model computation at query time — far too slow to run against an entire corpus. This is exactly why reranking is always a second stage, applied only to a small candidate pool that a cheaper method has already narrowed down.
- Why this is the natural next step: earlier topics built increasingly good candidate retrieval, and diversity-aware re-selection came after that. This topic adds the missing piece — a much more accurate relevance judgment, applied to a short list, as a final quality gate. The pattern across this whole chapter has consistently been: cheap-and-broad first, expensive-and-precise last, applied only to what's already been narrowed down — reranking is the purest expression of that principle.


### 2. Internal Working — Step by Step

**Bi-encoder scoring (what earlier topics already built):**

- Query goes through an encoder once, producing a vector.
- Each document goes through the same encoder once, at ingest time, producing its own vector.
- The relevance score is just a cheap comparison (like a dot product) between the two pre-computed vectors.

**Cross-encoder scoring (this topic):**

- The query and one candidate document are combined into a single input together.
- That combined input goes through the model, and the model produces one single relevance score for that specific pair.
- This has to be repeated separately for every candidate document being considered against that query — there's no way to precompute anything ahead of time, because the model needs both pieces together as input.

**Step-by-step reranking pipeline:**

1. Run cheap retrieval first (hybrid BM25 + dense + RRF) to get a candidate pool — typically a few dozen candidates.
2. For each candidate document in the pool, pair it together with the query as a single combined input.
3. Pass every pair through the cross-encoder model, ideally in efficient batches rather than one at a time.
4. Each pair receives its own independent relevance score. Unlike bi-encoder scores, cross-encoder scores for the same query are directly comparable across different candidate documents (though generally not comparable across entirely different queries).
5. Sort candidates by this cross-encoder score, highest first.
6. Return only the top few as the final result set that actually gets used downstream.

**Why cross-encoder scores are more accurate — the actual mechanism:**

- Because the query and document are processed together, the model's internal attention mechanism can directly compute something like "how well does this specific word in the query align with this specific word in this specific document, given the surrounding context of both?"
- A bi-encoder's fixed vector for a document was computed without ever knowing what the query would be — it's a general-purpose summary, not a summary tailored to any particular question.
- This is the same fundamental idea discussed with ColBERT (Topic 3): more interaction between query and document tends to mean higher quality but lower speed. A cross-encoder takes this to its logical extreme — full interaction, every part of the query attending to every part of the document — at the cost of having to redo this full computation for every single candidate, for every single query.


### 3. How This Is Implemented in This Project

- Reranking fits into the pipeline right after the initial retrieval and fusion step: hybrid retrieval produces a wider candidate pool, the cross-encoder reranker re-scores every candidate in that pool and narrows it down to a much smaller top-k by its own more accurate relevance judgment, and (optionally) a diversity step can then be applied on that smaller, more accurately-scored set before the final chunks are handed off downstream.
- There are two broad practical choices for the reranking model itself: a locally-run open-source cross-encoder model (runs on local hardware, no per-query API cost, and multilingual variants exist — important for a majority-Hinglish corpus), or a managed reranking API service (no local compute needed, often strong quality, but introduces a per-query cost, network latency, and — importantly — means sending the actual query and document content to a third-party service, which is a real consideration if that content contains customer PII).
- The core code pattern for a local cross-encoder: build a list of [query, candidate_text] pairs for every candidate in the pool, pass that whole list through the model in one batched call (much more efficient than one call per pair), attach the resulting scores back to each candidate, then sort by that score and keep only the top few.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Latency is the central trade-off:** a cross-encoder has to run a full model computation per candidate, per query — this is dramatically slower than a bi-encoder's single cheap comparison. This is exactly why reranking is only ever applied to a small, pre-narrowed candidate pool, never to an entire corpus.
- **Batching matters a lot for throughput:** running the reranker over all candidates in one efficient batched call is far faster than calling it once per candidate in a loop, since these models are built to take advantage of batched, parallel computation.
- **Choosing candidate pool size before reranking:** too small a pool risks the correct document never even making it to the reranking stage in the first place (the reranker can only rescue what earlier retrieval already found); too large a pool defeats the latency benefit of only reranking a short list. This needs to be tuned based on measured recall at the earlier retrieval stage.
- **Cost, for API-based rerankers specifically:** a managed reranking API typically charges per query or per document reranked, which adds an ongoing operating cost proportional to traffic volume — worth comparing against the compute cost of running an open-source reranker locally.
- **Security and privacy for API-based rerankers:** sending customer email content and candidate documents to an external API means that content leaves the local environment — this is a real consideration whenever the content includes personally identifiable information, and should be evaluated with the same care as any other third-party data-sharing decision.
- **Monitoring:** track how often the reranker's top-1 result differs from the earlier retrieval stage's top-1 result — a high disagreement rate suggests the reranker is adding real value by correcting the initial ranking; a very low disagreement rate might mean the reranking step isn't contributing much for the current query distribution, and its cost should be reconsidered.
- **Debugging a bad reranked result:** first confirm the correct document was actually present in the candidate pool handed to the reranker at all — if it wasn't, this is an earlier retrieval-stage problem, not a reranking problem, and no reranker can rescue a document that never made it into its input. If the document was present but scored poorly by the reranker, check whether the reranker model itself is well-suited to the corpus's language and domain — a reranker trained mostly on one language may score multilingual or code-mixed text poorly.
- **Deployment:** a locally-run reranker needs to be loaded into memory once and reused across requests, similar to an embedding model. An API-based reranker requires handling network calls, potential rate limits, and retries as part of the request pipeline.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Bi-encoder-only vs adding a reranking stage:** a bi-encoder-only pipeline is faster and cheaper but less accurate at fine-grained relevance judgments; adding a reranking stage improves accuracy at the cost of extra latency and compute, applied only to a short candidate list to keep that cost bounded. The decision to add reranking should follow from measuring that the earlier retrieval stage's ranking quality is genuinely insufficient, not from assuming a more sophisticated stage is automatically better.
- **Local open-source reranker vs managed API reranker:** a local reranker avoids ongoing per-query costs and keeps all data in the local environment, at the cost of needing local compute resources and model maintenance. A managed API reranker needs no local compute and may offer strong out-of-the-box quality, at the cost of ongoing usage fees, network latency, and sending content to a third party — a meaningful trade-off when the content includes sensitive customer data.
- **Where reranking sits relative to diversity re-ranking (MMR, from the previous topic):** reranking should generally run before a diversity step like MMR, not after — this way, MMR's relevance term is based on the more accurate reranked score, rather than the less accurate earlier retrieval score. Running diversity re-ranking first and reranking second would waste the reranker's precision on an already-diversity-adjusted (and potentially less relevant) list.
- **How large a candidate pool to rerank:** a larger pool increases the chance the correct document is included, but increases the total reranking latency and cost proportionally, since every additional candidate needs its own full model computation. This should be tuned based on measured recall from the earlier retrieval stage, not chosen arbitrarily.


### 6. Alternatives and When to Use Each

- **No reranking stage (retrieval scores used directly):** appropriate when the earlier retrieval stage's ranking quality is already good enough for the use case, or when latency and cost constraints are strict enough that the extra reranking step isn't affordable.
- **Local open-source cross-encoder reranker:** appropriate when data privacy is a priority, when there's local compute available, and when avoiding ongoing per-query costs matters — especially relevant for a majority non-English corpus where a multilingual reranker model is needed.
- **Managed reranking API:** appropriate when local compute isn't available or convenient, when the team wants strong quality without maintaining a model, and when the content being reranked doesn't raise significant data-sharing concerns.
- **ColBERT-style late interaction (Topic 3), as a middle ground:** offers some of the accuracy benefit of true cross-encoders while still allowing partial precomputation, at the cost of more storage — worth considering if pure bi-encoder retrieval is measurably insufficient but full cross-encoder reranking's latency is too costly at the required candidate pool size.


### 7. Common Mistakes and Production Failures

- Running a cross-encoder reranker over an entire corpus instead of a small, pre-narrowed candidate pool — this defeats the entire design principle behind reranking and creates a severe latency problem at any meaningful corpus size.
- Calling the reranker once per candidate in a loop instead of using an efficient batched call — this wastes most of the model's potential throughput.
- Choosing too small a candidate pool before reranking, such that the actually-correct document never makes it into the pool the reranker sees — no reranker can recover a document it was never given the chance to score.
- Sending sensitive customer content to a third-party reranking API without first considering the same data-sharing and privacy implications that would apply to any other external service handling that data.
- Assuming a reranker trained primarily on one language will perform equally well on a majority non-English or code-mixed corpus without verifying this on real evaluation data first.
- Applying diversity re-ranking (MMR) before reranking instead of after, wasting the reranker's more accurate relevance judgment on a list that's already been reordered based on a less accurate earlier score.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What's the fundamental architectural difference between a bi-encoder and a cross-encoder?
  A: A bi-encoder encodes the query and document completely independently, then compares their two fixed representations afterward with a cheap operation. A cross-encoder takes the query and a specific document together, as one combined input, letting the model directly compare them token by token, and outputs a single relevance score for that specific pair.

- Q: Why can't a cross-encoder be used to search an entire corpus directly, the way a bi-encoder can?
  A: A cross-encoder needs both the query and a specific document together as input, so nothing can be precomputed ahead of time — every single candidate has to go through a full model computation for every single query. This is far too slow to apply to an entire corpus, which is why it's only ever used as a second stage on a small, pre-narrowed candidate pool.

**Intermediate**

- Q: Why are cross-encoder scores for the same query directly comparable across different candidates, while bi-encoder scores from different retrieval methods might not be?
  A: A cross-encoder produces its score by jointly considering the same query alongside each specific candidate, using the same model in the same way each time — so all its scores for that one query are on a consistent, comparable scale. Comparing scores from fundamentally different scoring systems (like a keyword-based score and a semantic similarity score) doesn't have this property, which is exactly the incompatibility problem that motivated using rank-based fusion earlier in this chapter.

- Q: What's the main risk of choosing too small a candidate pool before reranking?
  A: The reranker can only rescore what it's given — if the genuinely correct document didn't make it into the candidate pool from the earlier retrieval stage, no amount of reranking accuracy can recover it. This means reranking quality is fundamentally capped by the recall of the earlier retrieval stage.

**Advanced**

- Q: A teammate wants to skip earlier retrieval methods entirely and just use a cross-encoder to score every document in the corpus for every query, since it's more accurate. How do you respond?
  A: Accuracy alone doesn't make this workable — a cross-encoder requires a full model computation per (query, document) pair, with no ability to precompute anything ahead of time. At any meaningful corpus size, this would make search latency completely impractical. The correct architecture keeps cheap, precomputable retrieval methods to narrow down the corpus to a small candidate pool first, and reserves the cross-encoder's accuracy for that much smaller, already-narrowed set.

- Q: How would you decide between a local open-source reranker and a managed reranking API for a project handling sensitive customer data?
  A: The primary consideration is where the data goes — a managed API means the query and candidate document content is sent to a third-party service, which needs to be evaluated with the same scrutiny as any other external data-sharing decision, especially if that content includes personally identifiable information. A local reranker keeps all content within the local environment at the cost of needing local compute resources and ongoing model maintenance. The choice should be driven by the sensitivity of the actual content being processed, not just by comparing raw model quality.

**Scenario-based**

- Q: After adding a reranking stage, you observe barely any change in the final top-1 result compared to before. How do you interpret this, and what would you check?
  A: This could mean the earlier retrieval and fusion stage was already producing a strong top-1 result for the current query distribution, making the reranker's extra accuracy less impactful in practice — in which case the added latency and cost of reranking should be weighed against this modest benefit. Alternatively, it could mean the reranker model itself isn't well-suited to this corpus's language or domain and isn't meaningfully distinguishing between candidates. The way to tell the difference is to look at the actual reranker scores across candidates — if they're all very close together, the reranker may not be discriminating well; if they show clear separation but the top document rarely changes, the earlier stage was likely already doing a good job.

**Follow-up questions to expect:**

- "How would you measure whether reranking is actually improving downstream answer quality, not just changing the order for its own sake?"
- "What would you do if reranking latency became a bottleneck at higher query volume?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **The interaction-speed trade-off, revisited:** this is the same spectrum introduced with ColBERT — full cross-attention (cross-encoders) sits at the highest-accuracy, lowest-speed end; bi-encoders with no interaction at all sit at the fastest, least fine-grained end; ColBERT's per-token late interaction sits in between. Recognizing this shared spectrum across different retrieval architectures is a strong signal of genuine understanding versus memorized facts about each individual method.
- **Why cross-encoder scores aren't necessarily comparable across different queries:** because the model jointly processes the query and document together, the resulting score reflects a relationship specific to that pairing, not a universal, query-independent measure of document quality — this means thresholds or comparisons across entirely different queries should be treated cautiously.
- **Multi-stage retrieval pipelines as a design pattern:** this "cheap-and-broad first, expensive-and-precise last, only on what's already narrowed down" pattern isn't unique to search — it shows up broadly in systems that need to balance thoroughness against cost, wherever an expensive precise method exists alongside a cheaper approximate one.

### 10. Quick Revision Summary (for last-minute interview prep)

> A cross-encoder solves the accuracy limitation shared by every prior retrieval method in this chapter: bi-encoder methods (BM25, dense retrieval, hybrid RRF) encode the query and each document completely independently, so the model never actually sees them together and can't model fine-grained interactions between specific words. A cross-encoder takes the query and one candidate document together as a single joint input, letting the model directly compare them and produce a much more accurate relevance score for that specific pair — at the cost of needing a full model computation per candidate per query, with nothing precomputable ahead of time. This is exactly why reranking is always a second stage applied to a small, already-narrowed candidate pool, never to an entire corpus. The practical choice is between a local open-source reranker (no ongoing per-query cost, keeps data local, needs local compute) and a managed reranking API (no local compute needed, but ongoing cost and a real data-privacy consideration since content is sent to a third party). Reranking should generally run before any diversity-based re-ranking step, and its overall quality is fundamentally capped by whatever candidate pool the earlier retrieval stage actually provides it.


### Module 1: Setup — Bi-Encoder Baseline

A real cross-encoder transformer needs internet access to download (not available here). We build a genuine, working baseline (bi-encoder cosine similarity, same TF-IDF+SVD approach as Topic 2) and then, in Module 2, a simplified but REAL joint-scoring function that stands in for a cross-encoder's core idea: looking at query and document TOGETHER instead of separately.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Bi-encoder baseline (same approach as Topic 2)
# ------------------------------------------------------------------

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

CANDIDATE_POOL = [
    "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
    "Fixed Deposit premature closure is allowed subject to applicable penalty charges as per RBI guidelines.",
    "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum.",
    "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
    "Nomination is available for FD accounts but does not affect the premature withdrawal penalty amount.",
    "Early exit from your deposit account will attract a deduction from your returns.",
]

QUERY = "what exact percentage penalty applies if I withdraw my FD before maturity"

def normalize_vector(v: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

vectorizer = TfidfVectorizer()
sparse_matrix = vectorizer.fit_transform(CANDIDATE_POOL)
svd = TruncatedSVD(n_components=5, random_state=42)
doc_vectors = svd.fit_transform(sparse_matrix)
doc_vectors_norm = np.array([normalize_vector(v) for v in doc_vectors])

query_vector = normalize_vector(svd.transform(vectorizer.transform([QUERY]))[0])
bi_encoder_scores = np.array([cosine_similarity(query_vector, v) for v in doc_vectors_norm])
bi_encoder_ranking = np.argsort(bi_encoder_scores)[::-1]

print("=" * 70)
print("MODULE 1: BI-ENCODER BASELINE (independent encoding)")
print("=" * 70)
print(f"Query: '{QUERY}'\n")
for rank, idx in enumerate(bi_encoder_ranking, start=1):
    print(f"  Rank {rank} | Doc {idx} | score={bi_encoder_scores[idx]:.4f} | {CANDIDATE_POOL[idx][:60]}...")

print("\nEach document's vector was computed WITHOUT knowledge of this")
print("query -- it is a general-purpose summary, fixed ahead of time.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: BI-ENCODER BASELINE (independent encoding)
Query: 'what exact percentage penalty applies if I withdraw my FD before maturity'

  Rank 1 | Doc 4 | score=0.9245 | Nomination is available for FD accounts but does not affect ...
  Rank 2 | Doc 0 | score=0.8644 | Premature withdrawal of FD incurs a 1 percent penalty on the...
  Rank 3 | Doc 1 | score=0.3051 | Fixed Deposit premature closure is allowed subject to applic...
  Rank 4 | Doc 3 | score=0.0851 | Senior citizens receive an additional 0.5 percent interest o...
  Rank 5 | Doc 2 | score=0.0125 | The Fixed Deposit interest rate for 24 months is 7.1 percent...
  Rank 6 | Doc 5 | score=-0.0089 | Early exit from your deposit account will attract a deductio...

Each document's vector was computed WITHOUT knowledge of this
query -- it is a general-purpose summary, fixed ahead of time.

Module 1 complete. Run Module 2 next.


### Module 2: Cross-Encoder-Style Joint Scoring

A simplified but REAL joint-interaction function that looks at the query and each document TOGETHER — the core idea a real cross-encoder implements with a transformer, done here with a much simpler (but genuinely joint, not precomputable) mechanism.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Cross-encoder-style joint scoring
#
# A real cross-encoder feeds [query, document] together through a
# transformer and lets attention compare every token pair directly.
# We cannot run a transformer here, but we CAN build a genuinely
# JOINT scoring function -- one that requires both query and document
# together as input, cannot be precomputed ahead of time, and directly
# rewards exact phrase-level alignment the bi-encoder's fixed vectors
# compress away. This is a simplified stand-in for the MECHANISM,
# not a claim that it matches real cross-encoder accuracy.
# ------------------------------------------------------------------

def longest_common_word_run(query: str, document: str) -> int:
    """Finds the length of the longest run of consecutive words that
    appear in the same order in both query and document. This is a
    JOINT feature -- it cannot be computed from separate fixed vectors,
    only by directly comparing the two texts together, which is the
    core property that makes a cross-encoder more precise."""
    q_words = query.lower().split()
    d_words = document.lower().split()
    best = 0
    for i in range(len(q_words)):
        for j in range(len(d_words)):
            length = 0
            while (i + length < len(q_words) and j + length < len(d_words)
                   and q_words[i + length] == d_words[j + length]):
                length += 1
            best = max(best, length)
    return best


def joint_cross_encoder_style_score(query: str, document: str) -> float:
    """Combines the bi-encoder-style overlap ratio with a JOINT phrase-
    alignment bonus -- rewarding documents that share exact, in-order
    phrasing with the query, which fixed independent vectors tend to
    blur together into a general 'somewhat similar' signal."""
    q_words = set(query.lower().split())
    d_words = set(document.lower().split())
    overlap_ratio = len(q_words & d_words) / len(q_words) if q_words else 0.0
    phrase_bonus = longest_common_word_run(query, document) / len(query.split())
    return overlap_ratio + (2.0 * phrase_bonus)  # phrase alignment weighted heavily


cross_encoder_scores = np.array([
    joint_cross_encoder_style_score(QUERY, doc) for doc in CANDIDATE_POOL
])
cross_encoder_ranking = np.argsort(cross_encoder_scores)[::-1]

print("=" * 70)
print("MODULE 2: CROSS-ENCODER-STYLE JOINT SCORING")
print("=" * 70)
print(f"Query: '{QUERY}'\n")
for rank, idx in enumerate(cross_encoder_ranking, start=1):
    run_length = longest_common_word_run(QUERY, CANDIDATE_POOL[idx])
    print(f"  Rank {rank} | Doc {idx} | score={cross_encoder_scores[idx]:.4f} | "
          f"longest matching phrase={run_length} words | {CANDIDATE_POOL[idx][:55]}...")

print(f"\nBi-encoder top-1:       Doc {bi_encoder_ranking[0]}")
print(f"Cross-encoder-style top-1: Doc {cross_encoder_ranking[0]}")
if bi_encoder_ranking[0] != cross_encoder_ranking[0]:
    print("\nThe two methods DISAGREE on the top result. The joint scorer")
    print("can directly reward exact, in-order phrase alignment with the")
    print("query, something the bi-encoder's compressed, query-unaware")
    print("vector could underweight relative to overall topical similarity.")
else:
    print("\nBoth methods agree on the top result here -- reranking's value")
    print("shows up more clearly further down the ranking or on harder,")
    print("more ambiguous queries where topical similarity alone is")
    print("insufficient to find the exact best match.")
print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: CROSS-ENCODER-STYLE JOINT SCORING
Query: 'what exact percentage penalty applies if I withdraw my FD before maturity'

  Rank 1 | Doc 4 | score=0.3333 | longest matching phrase=1 words | Nomination is available for FD accounts but does not af...
  Rank 2 | Doc 0 | score=0.3333 | longest matching phrase=1 words | Premature withdrawal of FD incurs a 1 percent penalty o...
  Rank 3 | Doc 1 | score=0.2500 | longest matching phrase=1 words | Fixed Deposit premature closure is allowed subject to a...
  Rank 4 | Doc 5 | score=0.0000 | longest matching phrase=0 words | Early exit from your deposit account will attract a ded...
  Rank 5 | Doc 2 | score=0.0000 | longest matching phrase=0 words | The Fixed Deposit interest rate for 24 months is 7.1 pe...
  Rank 6 | Doc 3 | score=0.0000 | longest matching phrase=0 words | Senior citizens receive an additional 0.5 percent inter...

Bi-encoder top-1:       Doc 4
Cross-encoder-style top-1: Doc 4

Both methods agree on the top result here -- 

### Module 3: Why This Cannot Scale to a Full Corpus

Measures the actual computational cost difference between the bi-encoder's precomputable approach and the joint scorer's per-pair-at-query-time approach.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: The scaling cost of joint (cross-encoder-style) scoring
# ------------------------------------------------------------------

import time

def time_bi_encoder_search(query_vec, doc_vecs, n_repeats=200):
    start = time.perf_counter()
    for _ in range(n_repeats):
        _ = [cosine_similarity(query_vec, v) for v in doc_vecs]
    return (time.perf_counter() - start) / n_repeats


def time_joint_scoring(query, documents, n_repeats=200):
    start = time.perf_counter()
    for _ in range(n_repeats):
        _ = [joint_cross_encoder_style_score(query, d) for d in documents]
    return (time.perf_counter() - start) / n_repeats


bi_encoder_time = time_bi_encoder_search(query_vector, doc_vectors_norm)
joint_time = time_joint_scoring(QUERY, CANDIDATE_POOL)

print("=" * 70)
print("MODULE 3: COMPUTATIONAL COST -- precomputed vs joint scoring")
print("=" * 70)
print(f"Candidate pool size: {len(CANDIDATE_POOL)} documents\n")
print(f"Bi-encoder search time (vectors already precomputed): {bi_encoder_time*1000:.4f} ms")
print(f"Joint (cross-encoder-style) scoring time (computed at query time): {joint_time*1000:.4f} ms")
print(f"Joint scoring is approximately {joint_time/bi_encoder_time:.1f}x slower per query")
print("at this tiny candidate pool size, even with our SIMPLIFIED joint")
print("function -- a real transformer-based cross-encoder is dramatically")
print("more expensive per pair than this toy example, and that gap widens")
print("with document length and model size.")

print("\nThe crucial point: bi-encoder document vectors are computed ONCE,")
print("regardless of how many queries arrive later. Joint/cross-encoder")
print("scoring must be redone for EVERY (query, document) pair, EVERY time.")
print("This is why cross-encoders only ever run on a small, pre-narrowed")
print("candidate pool -- never on an entire corpus.")

print(f"\nAt corpus size 6:      6 joint computations per query.")
print(f"At corpus size 500,000: 500,000 joint computations per query --")
print("completely impractical without first narrowing down with cheap")
print("retrieval (BM25 + dense + RRF from Topics 1, 2, 4).")

print("\nModule 3 complete. All Topic 7 modules done.")
print()
print("=" * 70)
print("CHAPTER 7 TOPIC 7 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Bi-encoder: query and document encoded INDEPENDENTLY, compared
  afterward. Fast, precomputable, but cannot model fine-grained
  query-document interaction.

  Cross-encoder: query and document processed TOGETHER as one input.
  Much more accurate relevance judgment, but cannot be precomputed --
  requires a full computation per (query, document) pair, per query.

  This is exactly why reranking is a SECOND STAGE applied to a small,
  already-narrowed candidate pool -- never to an entire corpus.

  Reranking should run BEFORE diversity re-ranking (MMR, Topic 6),
  so MMR's relevance term uses the more accurate reranked score.

  Reranking quality is capped by whatever candidate pool the earlier
  retrieval stage actually provides -- it cannot rescue a document
  that was never retrieved in the first place.
""")


MODULE 3: COMPUTATIONAL COST -- precomputed vs joint scoring
Candidate pool size: 6 documents

Bi-encoder search time (vectors already precomputed): 0.0290 ms
Joint (cross-encoder-style) scoring time (computed at query time): 0.3092 ms
Joint scoring is approximately 10.7x slower per query
at this tiny candidate pool size, even with our SIMPLIFIED joint
function -- a real transformer-based cross-encoder is dramatically
more expensive per pair than this toy example, and that gap widens
with document length and model size.

The crucial point: bi-encoder document vectors are computed ONCE,
regardless of how many queries arrive later. Joint/cross-encoder
scoring must be redone for EVERY (query, document) pair, EVERY time.
This is why cross-encoders only ever run on a small, pre-narrowed
candidate pool -- never on an entire corpus.

At corpus size 6:      6 joint computations per query.
At corpus size 500,000: 500,000 joint computations per query --
completely impractical without first narro